# Live Prediction: Free Power Hours in the Next 7 Days

This notebook shows the model's live predictions as a chart that refreshes itself.

**Pipeline:**
`Open-Meteo forecast` → `python -m src.ingest.stream live` → Kafka `openmeteo.forecast` → `python -m src.predict` (Spark Structured Streaming + saved model) → Kafka `predictions.cheap_hours` → **this notebook**

**Before running:** start both jobs in two terminals (project root):

```bash
python -m src.ingest.stream live --interval 900
python -m src.predict
```

The chart shows, per hour (Vienna time):
- **Bars:** predicted probability that the day-ahead price is ≤ 0 €/MWh; green = above the threshold ("free power hour")
- **Black line:** the actual aWATTar price, where already known (tomorrow's prices are published around 13:00)
- **Grey area:** day 4 onwards, where weather forecasts become noticeably less reliable

Section 4 replays a **past summer week** the model has never seen, to show the model on a week with many negative prices.

## 1. Setup

In [ ]:
import json
import sys
import time
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import clear_output, display
from confluent_kafka import Consumer, TopicPartition

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))   # so `import src...` works from notebooks/
from src.ingest.config import KAFKA_BOOTSTRAP, TOPICS

TZ = "Europe/Vienna"
PRED_TOPIC = "predictions.cheap_hours"
DAYS = 7               # chart window
RELIABLE_DAYS = 3      # weather forecasts are good for ~3 days, then get less reliable
REFRESH_SECONDS = 60
print("Kafka:", KAFKA_BOOTSTRAP)

## 2. Reading from Kafka

`TopicReader` keeps one consumer per topic open and, on every refresh, only reads the **new** messages since the last call. It starts at messages written in the last few days (Kafka can look up offsets by time), so it doesn't replay the whole history.

Every topic can contain the same key many times (each forecast poll re-scores the same hours, prices can be corrected). As in the data contract, we keep the **latest message per key**.

In [ ]:
class TopicReader:
    """Incrementally read a JSON topic, keeping the latest value per `key_field`."""

    def __init__(self, topic, key_field="ts", lookback_days=3):
        self.topic, self.key_field, self.latest = topic, key_field, {}
        self.consumer = Consumer({"bootstrap.servers": KAFKA_BOOTSTRAP,
                                  "group.id": f"live-chart-{topic}-{time.time_ns()}",
                                  "enable.auto.commit": False})
        partitions = self.consumer.list_topics(topic, timeout=10).topics[topic].partitions
        since_ms = int((time.time() - lookback_days * 86400) * 1000)
        offsets = self.consumer.offsets_for_times(
            [TopicPartition(topic, p, since_ms) for p in partitions], timeout=10)
        # offset -1 = no message after that time -> start at the end
        self.consumer.assign([TopicPartition(topic, tp.partition, tp.offset if tp.offset >= 0 else -1)
                              for tp in offsets])

    def poll(self, max_seconds=5.0):
        """Read everything new; return a DataFrame with the latest row per key."""
        deadline = time.time() + max_seconds
        while time.time() < deadline:
            msgs = self.consumer.consume(num_messages=5000, timeout=0.5)
            if not msgs:
                break
            for m in msgs:
                if m.error() is None:
                    rec = json.loads(m.value())
                    self.latest[rec[self.key_field]] = rec
        df = pd.DataFrame(self.latest.values())
        if not df.empty:
            df["ts"] = pd.to_datetime(df["ts"], utc=True).dt.tz_convert(TZ)
            df = df.sort_values("ts").reset_index(drop=True)
        return df


predictions = TopicReader(PRED_TOPIC)
prices = TopicReader(TOPICS["awattar"])
pred_df, price_df = predictions.poll(), prices.poll()
print(f"{len(pred_df)} predicted hours, {len(price_df)} price hours")
pred_df.tail()

## 3. The chart

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch
from matplotlib.ticker import PercentFormatter

C = {  # colour palette
    "text": "#111827", "muted": "#6b7280", "grid": "#e5e7eb", "card": "#f8fafc",
    "free": "#10b981", "free_dark": "#047857", "bar_lo": "#a5b4fc", "bar_hi": "#4f46e5",
    "price": "#f97316", "now": "#ef4444", "weekend": "#faf5ff", "unreliable": "#f1f5f9",
}
BAR_CMAP = LinearSegmentedColormap.from_list("prob", [C["bar_lo"], C["bar_hi"]])
STYLE = {"font.size": 11, "axes.edgecolor": C["grid"], "axes.labelcolor": C["muted"],
         "xtick.color": C["muted"], "ytick.color": C["muted"], "text.color": C["text"],
         "axes.spines.top": False, "axes.spines.right": False, "figure.facecolor": "white"}


def _pill(ax, x, y, text, color, transform=None, **kw):
    """Small rounded label, white text on a coloured background."""
    ax.text(x, y, text, color="white", fontsize=9, fontweight="bold", zorder=10,
            transform=transform or ax.transData,
            bbox=dict(boxstyle="round,pad=0.35,rounding_size=0.6", fc=color, ec="none"), **kw)


def _card(fig, x, w, label, value, sub, color):
    """KPI card in figure coordinates: big value, small caption."""
    fig.patches.append(FancyBboxPatch((x, 0.795), w, 0.115, transform=fig.transFigure,
                                      boxstyle="round,pad=0,rounding_size=0.012",
                                      fc=C["card"], ec=C["grid"], lw=1, zorder=0))
    fig.text(x + 0.012, 0.888, label.upper(), fontsize=9, fontweight="bold", color=C["muted"], va="center")
    fig.text(x + 0.012, 0.847, value, fontsize=21, fontweight="bold", color=color, va="center")
    fig.text(x + 0.012, 0.810, sub, fontsize=9, color=C["muted"], va="center")


def _backgrounds(ax, start, end, unreliable):
    for d in range((end - start).days):
        day = start + pd.Timedelta(days=d)
        if day.day_of_week >= 5:
            ax.axvspan(day, day + pd.Timedelta(days=1), color=C["weekend"], zorder=0, lw=0)
    ax.axvspan(unreliable, end, color=C["unreliable"], zorder=0, lw=0)


def plot_week(pred_df, price_df, replay_start=None):
    """Live mode: the next DAYS days from today. Replay mode: a past week starting at `replay_start`."""
    replay = replay_start is not None
    now = pd.Timestamp.now(tz=TZ)
    start = pd.Timestamp(replay_start).tz_convert(TZ).floor("D") if replay else now.floor("D")
    end = start + pd.Timedelta(days=DAYS)
    # Historical forecasts are all short-range, so no "less reliable" part in replay mode
    unreliable = end if replay else start + pd.Timedelta(days=RELIABLE_DAYS)

    with plt.rc_context(STYLE):
        fig = plt.figure(figsize=(15, 8.4))
        if pred_df.empty:
            fig.text(0.5, 0.5, "No predictions yet. Are `stream live` and `src.predict` running?",
                     ha="center", va="center", fontsize=14, color=C["muted"])
            return fig

        week = pred_df[(pred_df["ts"] >= start) & (pred_df["ts"] < end)].sort_values("ts")
        threshold = float(week["threshold"].iloc[-1]) if len(week) else 0.3
        free = week["p_negative"] >= threshold
        upcoming = week if replay else week[week["ts"] >= now.floor("h")]
        known = (price_df[(price_df["ts"] >= start) & (price_df["ts"] < end)].sort_values("ts")
                 if not price_df.empty else price_df)

        # ---------- header: title + KPI cards ----------
        gap, x0 = 0.015, 0.05
        w = (0.97 - x0 - 3 * gap) / 4   # four cards across the full chart width
        cards = [x0 + i * (w + gap) for i in range(4)]
        if replay:
            fig.text(0.05, 0.965, "Free Power Hours  ·  Replay", fontsize=22, fontweight="bold", va="center")
            fig.text(0.05, 0.93, f"Predictions from the weather forecasts of {start:%d.%m.} – "
                     f"{end - pd.Timedelta(days=1):%d.%m.%Y} vs. the actual prices  ·  week not used for training",
                     fontsize=11, color=C["muted"], va="center")
            merged = week.merge(known[["ts", "price_eur_mwh"]], on="ts", how="left")
            actual = merged["price_eur_mwh"] <= 0
            predicted = merged["p_negative"] >= threshold
            tp, n_act, n_pred = int((actual & predicted).sum()), int(actual.sum()), int(predicted.sum())
            _card(fig, cards[0], w, "Predicted free hours", str(n_pred), f"p ≥ {threshold:.0%} during the week",
                  C["bar_hi"])
            _card(fig, cards[1], w, "Actual hours ≤ 0 €", str(n_act), "aWATTar day-ahead price", C["free"])
            _card(fig, cards[2], w, "Caught", f"{tp / n_act:.0%}" if n_act else "–",
                  f"{tp} of {n_act} negative hours predicted", C["free"])
            _card(fig, cards[3], w, "Correct alarms", f"{tp / n_pred:.0%}" if n_pred else "–",
                  f"{tp} of {n_pred} predicted hours were ≤ 0 €", C["bar_hi"])
        else:
            fig.text(0.05, 0.965, "Free Power Hours", fontsize=22, fontweight="bold", va="center")
            issued = pd.to_datetime(week["forecast_issued_at"].max()).tz_convert(TZ) if len(week) else None
            fig.text(0.05, 0.93, f"Chance of a negative electricity price in Austria, next {DAYS} days"
                     + (f"  ·  forecast {issued:%a %d.%m. %H:%M}" if issued is not None else "")
                     + f"  ·  updated {now:%H:%M:%S}", fontsize=11, color=C["muted"], va="center")

        best = upcoming.loc[upcoming["p_negative"].idxmax()] if len(upcoming) else None
        if not replay:
            n_free = int((upcoming["p_negative"] >= threshold).sum())
            nxt = upcoming[upcoming["p_negative"] >= threshold].head(1)
            price_now = known.loc[known["ts"] == now.floor("h"), "price_eur_mwh"]
            _card(fig, cards[0], w, "Free power hours", str(n_free), f"p ≥ {threshold:.0%}, rest of the week",
                  C["free"] if n_free else C["muted"])
            _card(fig, cards[1], w, "Highest chance",
                  f"{best['p_negative']:.0%}" if best is not None else "–",
                  f"{best['ts']:%A %H:00}" if best is not None else "", C["bar_hi"])
            _card(fig, cards[2], w, "Next free hour",
                  f"{nxt['ts'].iloc[0]:%a %H:00}" if len(nxt) else "none",
                  "start charging the EV" if len(nxt) else "no hour above the threshold",
                  C["free"] if len(nxt) else C["muted"])
            _card(fig, cards[3], w, "Price right now",
                  f"{price_now.iloc[0]:.0f} €" if len(price_now) else "–", "per MWh, aWATTar day-ahead",
                  C["free"] if len(price_now) and price_now.iloc[0] <= 0 else C["price"])

        gs = fig.add_gridspec(2, 1, height_ratios=[3, 1.25], left=0.05, right=0.97,
                              top=0.75, bottom=0.08, hspace=0.12)
        ax = fig.add_subplot(gs[0])
        axp = fig.add_subplot(gs[1], sharex=ax)

        # ---------- top: probability per hour ----------
        _backgrounds(ax, start, end, unreliable)
        top = min(1.0, max(threshold * 1.6, week["p_negative"].max() * 1.25 if len(week) else 0))
        colors = [C["free"] if f else BAR_CMAP(min(p / threshold, 1.0))
                  for p, f in zip(week["p_negative"], free)]
        ax.bar(week["ts"], week["p_negative"], width=pd.Timedelta(minutes=48), align="edge",
               color=colors, zorder=3, lw=0)
        ax.axhline(threshold, color=C["free"], ls=(0, (5, 4)), lw=1.6, zorder=4)
        _pill(ax, start + pd.Timedelta(hours=3), threshold, f"threshold {threshold:.0%}", C["free"],
              ha="left", va="center")
        if replay and len(known):   # tick marks at the bottom: hours that really were <= 0 EUR
            neg = known[known["price_eur_mwh"] <= 0]["ts"]
            ax.scatter(neg + pd.Timedelta(minutes=24), [-0.025 * top] * len(neg), marker="^", s=28,
                       color=C["free_dark"], zorder=5, clip_on=False)
        if best is not None and best["p_negative"] > 0:
            ax.annotate(f"{best['p_negative']:.0%}", (best["ts"] + pd.Timedelta(minutes=24), best["p_negative"]),
                        xytext=(0, 8), textcoords="offset points", ha="center", fontsize=10,
                        fontweight="bold", color=C["free_dark"] if best["p_negative"] >= threshold else C["bar_hi"])
        ax.set_ylim(0, top)
        ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
        ax.set_ylabel("chance of price ≤ 0")
        ax.grid(axis="y", color=C["grid"], lw=0.8, zorder=1)
        ax.spines["left"].set_visible(False)
        ax.tick_params(axis="x", labelbottom=False, length=0)
        ax.tick_params(axis="y", length=0)
        from matplotlib.lines import Line2D
        from matplotlib.patches import Patch
        ax.legend(handles=[Patch(color=C["free"], label="free power hour (above threshold)"),
                           Patch(color=BAR_CMAP(0.6), label="chance below threshold"),
                           Patch(color=C["price"], alpha=0.6, label="actual price (bottom)")]
                          + ([Line2D([], [], marker="^", ls="", color=C["free_dark"], label="actual price ≤ 0")]
                             if replay else []),
                  loc="lower right", bbox_to_anchor=(1.0, 1.0), ncol=4, frameon=False, fontsize=10,
                  handlelength=1.2, borderaxespad=0.2)
        if not replay:
            _pill(ax, unreliable + pd.Timedelta(hours=3), 0.96, "less reliable forecast  →", "#94a3b8",
                  transform=ax.get_xaxis_transform(), va="top")

        # ---------- bottom: actual price ----------
        _backgrounds(axp, start, end, unreliable)
        if len(known):
            t = list(known["ts"]) + [known["ts"].iloc[-1] + pd.Timedelta(hours=1)]
            y = list(known["price_eur_mwh"]) + [known["price_eur_mwh"].iloc[-1]]
            axp.step(t, y, where="post", color=C["price"], lw=2, zorder=3)
            axp.fill_between(t, y, 0, step="post", where=[v > 0 for v in y], color=C["price"], alpha=0.15, lw=0)
            axp.fill_between(t, y, 0, step="post", where=[v <= 0 for v in y], color=C["free"], alpha=0.45, lw=0)
            lo, hi = min(y), max(y)
            axp.set_ylim(min(lo * 1.15, -15), max(hi * 1.15, 50))
            if t[-1] < end - pd.Timedelta(days=1):
                axp.text(t[-1] + pd.Timedelta(hours=3), (axp.get_ylim()[0] + axp.get_ylim()[1]) / 2,
                         "later prices are published\ndaily around 13:00", fontsize=9, color=C["muted"], va="center")
        else:
            axp.text(0.5, 0.5, "no prices yet", transform=axp.transAxes, ha="center", va="center", color=C["muted"])
        axp.axhline(0, color=C["muted"], lw=0.8, zorder=2)
        axp.set_ylabel("price €/MWh")
        axp.grid(axis="y", color=C["grid"], lw=0.8, zorder=1)
        axp.spines["left"].set_visible(False)
        axp.tick_params(axis="y", length=0)

        # ---------- shared x axis: one label per day, lines at midnight ----------
        tz = week["ts"].dt.tz
        noons = [start + pd.Timedelta(days=d, hours=12) for d in range(DAYS)]
        axp.set_xticks(noons, [f"{d:%a}\n{d:%d.%m.}" for d in noons])
        axp.tick_params(axis="x", length=0, labelsize=11, labelcolor=C["text"])
        for a in (ax, axp):
            a.xaxis.set_minor_locator(mdates.DayLocator(tz=tz))
            a.grid(axis="x", which="minor", color="white", lw=3, zorder=2)
            if not replay:
                a.axvline(now, color=C["now"], lw=2, zorder=6)
        ax.set_xlim(start, end)
        if not replay:
            _pill(ax, now, top, "now", C["now"], ha="center", va="center")
    return fig


def free_hours_table(pred_df):
    now = pd.Timestamp.now(tz=TZ).floor("h")
    up = pred_df[(pred_df["ts"] >= now) & pred_df["is_negative"]].sort_values("ts")
    return up.assign(hour=up["ts"].dt.strftime("%a %d.%m. %H:00"))[["hour", "p_negative"]] \
             .reset_index(drop=True)


fig = plot_week(pred_df, price_df)
plt.close(fig)
display(fig)

## 4. Replay: a summer week

In November there are hardly any negative prices, so the live chart is mostly empty. To show what the model does on a sunny week, we **replay a past summer week**:

- **Weather input:** `openmeteo.historical_forecast`, i.e. what the weather models predicted back then, the same kind of input the live model gets
- **Model:** the saved model from `models/cheap_hours`, unchanged
- **Week:** chosen from **July 2025 onwards**, which the model has **never seen** during training (training ended 30.06.2025). By default the summer week with the most negative-price hours; set `REPLAY_WEEK` to pick another Monday
- **Comparison:** the actual aWATTar prices of that week; ▲ marks the hours that really were ≤ 0 €

In [ ]:
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
from pyspark.sql import SparkSession, functions as F

from src.features import PRICE_SCHEMA, WEATHER_SCHEMA, parse_kafka, weather_features

REPLAY_WEEK = None          # e.g. "2025-08-11" (a Monday); None = summer week with most negative hours
TEST_START = "2025-07-01"   # first day the model has not seen in training
THRESHOLD = 0.3             # same default as src.predict
ISO = "yyyy-MM-dd'T'HH:mm:ssxxx"

spark = (SparkSession.builder.appName("replay-summer-week")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")


def read_topic(topic, schema):
    raw = (spark.read.format("kafka").option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
           .option("subscribe", topic).option("startingOffsets", "earliest").load())
    return parse_kafka(raw, schema)


all_prices = (read_topic(TOPICS["awattar"], PRICE_SCHEMA)
              .groupBy("ts").agg(F.max_by("price_eur_mwh", "ingested_at").alias("price_eur_mwh"))
              .cache())

# Summer weeks (Mon-Sun, Vienna time) the model has not seen, ranked by negative-price hours
local = F.from_utc_timestamp("ts", TZ)
weeks = (all_prices.filter(F.col("ts") >= TEST_START)
         .withColumn("local", local).filter(F.month("local").between(6, 8))
         .groupBy(F.date_format(F.date_trunc("week", "local"), "yyyy-MM-dd").alias("week"))
         .agg(F.count("*").alias("hours"),
              F.sum((F.col("price_eur_mwh") <= 0).cast("int")).alias("negative_hours"))
         .filter("hours >= 167")                     # complete weeks only (DST weeks have 167/169)
         .orderBy(F.desc("negative_hours")))
print("Unseen summer weeks with the most negative-price hours:")
weeks.show(5)
week_start = REPLAY_WEEK or weeks.first()["week"]

In [ ]:
start = pd.Timestamp(week_start, tz=TZ)
lo = F.lit(start.tz_convert("UTC").isoformat()).cast("timestamp")
hi = F.lit((start + pd.Timedelta(days=DAYS)).tz_convert("UTC").isoformat()).cast("timestamp")
in_week = (F.col("ts") >= lo) & (F.col("ts") < hi)

model = PipelineModel.load(str(PROJECT_ROOT / "models" / "cheap_hours"))
wx = read_topic(TOPICS["openmeteo_historical_forecast"], WEATHER_SCHEMA).filter(in_week)
scored = (model.transform(weather_features(wx))
          .select(F.date_format("ts", ISO).alias("ts"), vector_to_array("probability")[1].alias("p_negative"))
          .collect())
week_prices = all_prices.filter(in_week).select(F.date_format("ts", ISO).alias("ts"), "price_eur_mwh").collect()

# Spark rows -> pandas (timestamps as ISO strings, so the time zone is unambiguous)
replay_pred = pd.DataFrame(scored, columns=["ts", "p_negative"])
replay_pred["ts"] = pd.to_datetime(replay_pred["ts"], utc=True).dt.tz_convert(TZ)
replay_pred = replay_pred.assign(threshold=THRESHOLD, is_negative=replay_pred["p_negative"] >= THRESHOLD,
                                 forecast_issued_at=None).sort_values("ts")
replay_prices = pd.DataFrame(week_prices, columns=["ts", "price_eur_mwh"])
replay_prices["ts"] = pd.to_datetime(replay_prices["ts"], utc=True).dt.tz_convert(TZ)

fig = plot_week(replay_pred, replay_prices, replay_start=start)
plt.close(fig)
display(fig)

## 5. Live mode

Redraws the chart every minute with the newest predictions and prices. Stop it with the **Interrupt** button (■) of the notebook.

In [ ]:
def live(rounds=None):
    i = 0
    while rounds is None or i < rounds:
        pred_df, price_df = predictions.poll(), prices.poll()
        fig = plot_week(pred_df, price_df)
        plt.close(fig)
        clear_output(wait=True)
        display(fig)
        table = free_hours_table(pred_df) if not pred_df.empty else pd.DataFrame()
        if len(table):
            print("Upcoming free power hours:")
            display(table)
        else:
            print("No upcoming hours above the threshold.")
        i += 1
        if rounds is None or i < rounds:
            time.sleep(REFRESH_SECONDS)


live()